# CME 241 – Phase 1: Problem Definition  
## Multi‑Strategy Statistical Arbitrage for Personal Finance Optimization

---

## 1. Problem Specialization and Practical Relevance

In this project, we consider the problem of allocating capital to a collection of simple quantitative trading strategies (e.g., pairs trading, cross‑sectional momentum, and mean‑reversion) and dynamically reallocating capital among them and cash over time. Each individual strategy give a modest risk‑adjusted return and unstable performance, but the strategies are only partially correlated.

---

## 2. Markov Decision Process Formulation

We model the multi‑strategy allocation problem as an MDP $(S, A, P, R, \gamma)$, parameterized so that different settings correspond to three incrementally diluted versions of the same core problem.

### 2.1 State Space ${S}$

Time is discrete: $t = 0, 1, \dots, T$ (e.g., daily or weekly decision epochs).

At time $t$, the state is
$$
S_t = \big( W_t,\ \mathbf{w}_t,\ \hat{\boldsymbol{\mu}}_t,\ \hat{\boldsymbol{\sigma}}_t,\ R_t \big),
$$
where:

- $W_t \in \mathbb{R}_+$:  
  Current total invested in the strategy portfolio at time $t$.
- $\mathbf{w}_t = (w_t^{(1)}, \dots, w_t^{(K)})$:  
  Current capital allocations to each of $K$ trading strategies.
- $\hat{\boldsymbol{\mu}}_t = (\hat{\mu}_t^{(1)}, \dots, \hat{\mu}_t^{(K)})$:  
  Estimates of each strategy’s expected return, computed from recent data (e.g., rolling or exponentially weighted averages).
- $\hat{\boldsymbol{\sigma}}_t = (\hat{\sigma}_t^{(1)}, \dots, \hat{\sigma}_t^{(K)})$:  
  Estimates of each strategy’s volatility.
- $R_t$:  
  Optional discrete market‑regime indicator (e.g., low/medium/high volatility, bull/bear), following a small Markov chain.

---

### 2.2 Action Space $A$

At each time $t$, the agent chooses a new target allocation across strategies and cash:
$$
A_t = \mathbf{a}_t = \big( a_t^{(1)}, \dots, a_t^{(K)}, a_t^{\text{cash}} \big),
$$
where $a_t^{(k)}$ is the fraction of capital allocated to strategy $k$, and $a_t^{\text{cash}}$ is the fraction held in cash.

Constraints:

- **Budget constraint**:  
  $$
  \sum_{k=1}^{K} a_t^{(k)} + a_t^{\text{cash}} = 1.
  $$
- **Leverage constraint (if allowed)**:  
  $$
  \sum_{k=1}^{K} \big| a_t^{(k)} \big| \leq L,
  $$
  for some leverage limit $L \ge 1$.

In the Phase‑2 simplified version, $A$ will be a small discrete set of allocations (e.g., conservative/balanced/aggressive mixes). In the more realistic versions, $A$ will be continuous.

---

### 2.3 Transition Dynamics $P$

Let $r_{t+1}^{(k)}$ be the simple return of strategy $k$ from $t$ to $t+1$. The portfolio return is
$$
r_{p,t+1} = \sum_{k=1}^{K} a_t^{(k)} r_{t+1}^{(k)}.
$$

Let $\text{TC} (\mathbf{w}_t, \mathbf{a}_t)$ denote transaction costs (e.g., proportional to turnover) incurred when moving from the old allocation $\mathbf{w}_t$ to the new target $\mathbf{a}_t$. capital evolves as
$$
W_{t+1} = W_t \big(1 + r_{p,t+1}\big) - \text{TC}(\mathbf{w}_t, \mathbf{a}_t).
$$

Return dynamics depend on the version:

- **Simplified version (Phase 2)**:  
  Parametric model (e.g., Gaussian returns with fixed or regime‑dependent means, volatilities, and correlations).
- **Realistic version (Phase 3)**:  
  Returns driven by historical and/or block‑bootstrapped data from a panel of equities/ETFs; regime $R_t$ may be derived from realized volatility.

The statistics $\hat{\boldsymbol{\mu}}_t, \hat{\boldsymbol{\sigma}}_t$ are updated from realized strategy returns via rolling or exponential windows, and the regime $R_t$ evolves according to its transition matrix. Together, these define $P(S_{t+1} \mid S_t, A_t)$.

---

### 2.4 Reward Function $R$

The overall objective is to maximize aggregated Utility of Consumption; here we treat capital growth and risk as a proxy for future consumption capability. We will use one of the following equivalent reward formulations:

1. **Utility‑increment reward**  
   $$
   R(S_t, A_t, S_{t+1}) = U(W_{t+1}) - U(W_t),
   $$
   where $U$ is a concave utility function (e.g., CRRA).

2. **Risk‑adjusted return reward**  
   $$
   R(S_t, A_t, S_{t+1}) = r_{p,t+1} - \lambda \cdot \widehat{\text{Var}}(r_{p,t+1}),
   $$
   where $\lambda > 0$ is a risk‑aversion parameter, and the variance term is estimated from recent returns.

Both choices encourage higher returns while penalizing risk and turnover, consistent with personal‑finance tradeoffs between consumption level and uncertainty. [file:1]

---

### 2.5 Objective

For discount factor $\gamma \in (0,1]$, the objective under a policy $\pi$ is
$$
J^\pi = \mathbb{E}^\pi\left[ \sum_{t=0}^{T} \gamma^t R(S_t, A_t, S_{t+1}) \right].
$$

The goal is to find an optimal (or approximately optimal) policy $\pi^\star$ that maximizes $J^\pi$.

---

## 3. Three Versions

We define three versions of the problem: an ambitious “commercial” version, a diluted but realistic Phase‑3 version, and a further simplified Phase‑2 version. All three are parameterizations of a single `MarkovDecisionProcess` class.

---

### 3.1 Version 1 – Ambitious / “Commercial” Version

This is the version we would ideally like to solve with ample time and resources.

**Key features:**

- **Strategies and assets**  
  - Dozens to hundreds of simple statistical‑arbitrage alphas across stocks and ETFs: pairs trading, cross‑sectional momentum/value, mean‑reversion, volatility carry, etc.
- **State richness**  
  - Detailed capital and allocation per strategy.  
  - Per‑strategy performance statistics, factor exposures, and covariance structure (possibly via a factor model).  
  - Rich market‑regime indicators (macro features, volatility states, liquidity indicators).  
  - Realistic frictions: transaction costs, borrowing costs, margin and short‑selling constraints, potentially tax‑aware PnL to better integrate with personal‑finance decisions.
- **Dynamics**  
  - Non‑stationary, regime‑switching return processes calibrated from large historical datasets.
- **Solution method**  
  - Deep RL (e.g., recurrent actor–critic, distributional RL) trained on long simulated and historical paths, with risk constraints (e.g., volatility or drawdown limits) implemented via action constraints or penalties.

This version is too complex for the course timeline but represents the fully practical system envisioned by item (1) of the project guidance (commercially useful technology).

---

### 3.2 Version 2 – Phase 3 Version (Realistic RL)

This is the diluted yet realistic version we plan to solve in Phase 3 using RL, after building a simulated environment that generates sampling traces from the MDP, as required in the project description.

**Environment**

- Use daily returns for a small set of liquid ETFs or stocks (e.g., 5–10 tickers).
- Construct several simple strategies on top of this universe:  
  - A pairs trading strategy between two correlated assets.  
  - A cross‑sectional momentum strategy across the universe.  
  - A simple mean‑reversion strategy based on previous‑day returns or deviations from moving averages.  
  - Cash as a residual allocation.

**State:**

- capital $W_t$ and allocation fractions to each strategy and cash.  
- Rolling mean and volatility of each strategy’s recent returns (e.g., 20–60 day windows).  
- A simple regime derived from realized index volatility (e.g., low/medium/high).

**Action:**

- Continuous allocation vector $\mathbf{a}_t$ over the strategies and cash, with a modest leverage constraint.

**Reward:**

- Daily log‑utility of capital or risk‑adjusted return (return minus volatility and turnover penalties).

**Solution method:**

- Implement the environment as a `MarkovDecisionProcess` subclass and a wrapper that generates sampling traces from historical or block‑bootstrapped returns, exactly as described in the project guidance.
- Use an open‑source RL library (e.g., PPO, DDPG, SAC) to learn a policy, possibly with small customizations to better handle turnover and regime persistence, as encouraged in the LLM usage section.   
- Compare the learned policy against simple baselines (equal‑weight allocation, volatility‑weighted allocation, single‑strategy investments) to demonstrate that combining many weak, low‑correlated strategies yields a stronger aggregate strategy.

This version is “reasonably realistic (not too simplistic)” yet tractable within a few weeks, as requested for the Phase‑3 problem.

---

### 3.3 Version 3 – Phase 2 Version (Simplified DP/ADP)

This is the further diluted version to be solved using Dynamic Programming / Approximate Dynamic Programming in Phase 2, before applying RL, with deliberately small state and action spaces.

**Setup:**

- **Strategies**  
  - 2 risky strategies plus cash:
    - Strategy 1: conservative (lower mean, lower volatility).  
    - Strategy 2: aggressive (higher mean, higher volatility).  
- **Returns**  
  - Parametric Gaussian returns with specified means, volatilities, and correlation.  
  - Optionally regime‑dependent (good/bad regime) with a 2‑state Markov chain for $R_t$.
- **State**  
  - Capital $W_t$ discretized on a grid.  
  - Regime $R_t \in \{\text{good}, \text{bad}\}$ (if included), with a small known transition matrix.
- **Action**  
  - Small discrete set of allocations, e.g.:
    - Conservative: 80% in Strategy 1, 20% in Strategy 2.  
    - Balanced: 50% / 50%.  
    - Aggressive: 20% in Strategy 1, 80% in Strategy 2.  

**Reward and solution:**

- Reward: utility of capital per period or at terminal time.  
- Solution: formulate Bellman equations and solve via value iteration or policy iteration over the discretized state space.

This version is designed to:

- Be solvable with DP/ADP within reasonable computational limits.  
- Illustrate the tradeoff between model realism and DP tractability.  
- Highlight the “curse of dimensionality” and “curse of modeling,” motivating the move to RL in Phase 3, as described in the project guidance.

---

## 4. MarkovDecisionProcess Class Specification (Code‑Level Design)

To satisfy the requirement that all three versions be parameterizations of a single `MarkovDecisionProcess` class, we plan to implement:

```python
class MultiStrategyStatArbMDP(MarkovDecisionProcess):
    def __init__(
        self,
        n_strategies,
        horizon,
        initial_capital,
        return_model,
        transaction_cost_rate,
        leverage_limit,
        state_config,
        action_config,
        reward_config,
        rng_seed=None,
    ):
        """
        n_strategies: number of trading strategies (K)
        horizon: time horizon T
        initial_capital: initial capital W_0
        return_model: object specifying how to sample strategy returns
        transaction_cost_rate: parameter(s) for TC(w_t, a_t)
        leverage_limit: maximum allowed leverage
        state_config: controls which state components are included/updated
        action_config: discrete vs continuous actions, constraints
        reward_config: choice of utility / risk-adjusted reward, penalties
        """
        ...

    def initial_state(self):
        # Construct S_0 (W_0, initial allocations, statistics, regime)
        ...

    def step(self, state, action):
        # 1. Sample strategy returns from return_model
        # 2. Apply allocation and transaction costs to update capital
        # 3. Update statistics (mu_hat, sigma_hat) and regime
        # 4. Compute reward using reward_config
        # 5. Return next_state, reward, done_flag
        ...


In [7]:
file_path = r'crsp_daily_sp500.csv'
import pandas as pd


df = pd.read_csv(file_path)



/var/folders/zg/yv7tpd455rs9gt6r6451f5f80000gn/T/ipykernel_32557/4232191224.py:5: DtypeWarning: Columns (0: NAMEENDT, 1: SICCD, 2: NCUSIP, 3: SHRCLS, 4: TSYMBOL, 5: DCLRDT, 6: DLPDT, 7: NEXTDT, 8: SHRENDDT, 9: DLRETX, 10: DLRET, 11: RET, 12: RETX) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [6]:
print(df.head(100))

    PERMNO        date NAMEENDT  SHRCD  EXCHCD   SICCD NCUSIP TICKER  \
0    10137  1960-01-04      NaN   11.0     1.0  4910.0    NaN    NaN   
1    10145  1960-01-04      NaN   11.0     1.0  2810.0    NaN    NaN   
2    10225  1960-01-04      NaN   11.0     1.0  2110.0    NaN    NaN   
3    10401  1960-01-04      NaN   11.0     1.0  4810.0    NaN    NaN   
4    10516  1960-01-04      NaN   11.0     1.0  2030.0    NaN    NaN   
..     ...         ...      ...    ...     ...     ...    ...    ...   
95   21020  1960-01-04      NaN   11.0     1.0  4500.0    NaN    NaN   
96   21178  1960-01-04      NaN   11.0     1.0  3720.0    NaN    NaN   
97   21186  1960-01-04      NaN   11.0     1.0  2640.0    NaN    NaN   
98   21207  1960-01-04      NaN   11.0     1.0  6710.0    NaN    NaN   
99   21397  1960-01-04      NaN   11.0     1.0  2410.0    NaN    NaN   

                           COMNAM SHRCLS  ...     CFACSHR  OPENPRC NUMTRD  \
0           WEST PENN ELECTRIC CO    NaN  ...    4.000000 